### Script to generate Parquets

In [13]:
from pathlib import Path
import csv

from datasets import Dataset, Features, Audio, Value

# Change to Path("processed_data/Rutul") for the other dataset
root = Path("processed_data/Archi")

features = Features({
    "audio": Audio(sampling_rate=16000),
    "transcript": Value("string"),
})

for split in ["train", "test"]:
    rows = []

    with open(root / f"{split}.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            wav_path = root / row["audio"]

            rows.append({
                "audio": {
                    "path": row["audio"],  # optional metadata
                    "bytes": wav_path.read_bytes(),
                },
                "transcript": row["transcript"],
            })

    ds = Dataset.from_list(rows, features=features)

    out_dir = root / "parquet_data" / split
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "data.parquet"
    ds.to_parquet(str(out_file))

    size_mb = out_file.stat().st_size / (1024 * 1024)
    print(f"Saved {out_file} ({size_mb:.1f} MB)")

Creating parquet from Arrow format:   0%|          | 0/14 [00:00<?, ?ba/s]

Saved processed_data/Rutul/parquet_data/train/data.parquet (532.4 MB)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved processed_data/Rutul/parquet_data/test/data.parquet (49.5 MB)


In [ ]:
# Load model directly


In [20]:
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForCTC
import torch

# Streaming dataset
ds = load_dataset(
    "mahesh27/archi_rutul_asr",
    "Rutul",
    split="train",
    streaming=True,
)

# First example
sample = next(iter(ds))

processor = AutoProcessor.from_pretrained("mahesh27/w2v2l-custom-rutul")
model = AutoModelForCTC.from_pretrained(
    "mahesh27/w2v2l-custom-rutul",
    device_map="auto",
)

# Prepare input
inputs = processor(
    sample["audio"]["array"],
    sampling_rate=sample["audio"]["sampling_rate"],
    return_tensors="pt",
    padding=True,
)

# Move tensors to model device
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits

pred_ids = torch.argmax(logits, dim=-1)

transcription = processor.batch_decode(pred_ids)[0]

print("Reference :", sample["transcript"])
print("Prediction:", transcription)

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Reference : jeʃikbɨr qaʔas sejranda qʼulubɨr hɨwɨr ani jiʔi qʼulubɨr haʔ xur
Prediction: jeʃibɨ xasza sajrandiʃ ʃi hɨr ijiʔ  qʼlubr haʔ χur
